# Homework 4 — Statistics

**Deadline:** Monday, June 1, 2026, 23:59 (Barcelona time) &middot; **Total points:** 10 &middot; **Solo work**

This is your first hands-on with statistics in Python. Six parts.

You hand in this notebook with your code filled in, every `check_answer(...)` returning **PASS**, and the requested plots rendered.

The grader hides expected values — you see only PASS / FAIL.

**Libraries.** Use whatever you prefer for visualisation — `matplotlib`, `seaborn`, `plotly`, anything else. The grader only checks numerics — plots are for your understanding and for me to read.

**On AI.** Use it for syntax / docs / debugging. The math, the structure, and the interpretation should be yours. See the README.

---

## Setup

Run the cell below once. It loads the libraries and the hidden grader.

In [ ]:
# DO NOT MODIFY
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

from _grader import check_answer

np.random.seed(42)  # fix the seed for reproducible grading
print("ready")

---

## Part 1 — Generate and visualize distributions

We will sample from two random variables — one discrete, one continuous — and see how empirical pictures (histogram, PMF, ECDF) relate to the theoretical formulas you saw in class.

### Task 1a — Discrete random variable

Sample $X_1, \dots, X_n \sim \text{Bernoulli}(p)$ with $n = 1000$ and $p = 0.3$. Use `np.random.binomial(1, 0.3, size=n)` — `binomial(1, p)` is the same as `Bernoulli(p)`.

Plot the empirical PMF as a bar chart (two bars: at $0$ and at $1$, height = sample frequency). Compute the empirical mean and variance, compare with the theoretical ones:

- $\mathbb{E}[X] = p = 0.3$
- $\text{Var}(X) = p(1-p) = 0.21$

In [ ]:
np.random.seed(42)
n = 1000
p = 0.3

# Generate X_1, ..., X_n ~ Bernoulli(0.3) and store in X_bern
X_bern = ...

# Plot the empirical PMF as a bar chart (two bars at 0 and 1).

# Empirical mean and sample variance
mean_emp = ...
var_emp = ...

print(f"empirical mean = {mean_emp:.4f}  (theoretical {p})")
print(f"empirical var  = {var_emp:.4f}  (theoretical {p * (1 - p)})")

check_answer("1a_mean_emp", mean_emp)
check_answer("1a_var_emp", var_emp)

### Task 1b — Continuous random variable

Sample $X_1, \dots, X_n \sim \mathcal{N}(0, 1)$ with $n = 1000$. Use `np.random.normal(0, 1, size=n)`.

Plot a **density-normalized** histogram of the sample and overlay the theoretical PDF $\varphi(x) = \frac{1}{\sqrt{2\pi}} e^{-x^2/2}$ (`scipy.stats.norm.pdf`).

Compute the empirical mean and standard deviation. Compare with theoretical $\mu = 0$, $\sigma = 1$.

In [ ]:
np.random.seed(42)
n = 1000

# Generate X_1, ..., X_n ~ N(0, 1) and store in X_norm
X_norm = ...

# Plot a density-normalised histogram and overlay the theoretical PDF.

# Empirical mean and sample standard deviation
mean_emp = ...
std_emp = ...

print(f"empirical mean = {mean_emp:.4f}  (theoretical 0)")
print(f"empirical std  = {std_emp:.4f}  (theoretical 1)")

check_answer("1b_mean_emp", mean_emp)
check_answer("1b_std_emp", std_emp)

### Task 1c — Empirical CDF

For the continuous sample `X_norm` from 1b, plot the empirical CDF $F_n(x)$ as a step function and overlay the theoretical CDF $\Phi(x)$.

Definition: $F_n(x) = \frac{1}{n} \sum_{i=1}^n \mathbb{1}\{X_i \le x\}$. Easiest construction is to sort the sample and use $k/n$ as the height at the $k$-th sorted value.

Report the value of the ECDF at $x = 0$ — what fraction of the sample fell at or below $0$? The theoretical value is $\Phi(0) = 0.5$.

In [ ]:
# Build the empirical CDF of X_norm and overlay the theoretical Φ.

# Fraction of the sample at or below 0
ecdf_at_0 = ...

print(f"empirical F_n(0) = {ecdf_at_0:.4f}  (theoretical Φ(0) = 0.5)")

check_answer("1c_ecdf_at_0", ecdf_at_0)

---

## Part 2 — Estimating revenue: descriptive stats and a CI

You are a product analyst at a subscription product. The team needs a read on per-user revenue: a quick descriptive summary, a point estimate of the mean, and a confidence interval for that estimate so the PM knows how precise the number is.

`ab_data.csv` contains two columns: `control` (per-user revenue in the control bucket) and `treatment` (per-user revenue in the treatment bucket). For this part, work with the **control** group only.

Compute:

| Quantity | Definition |
|---|---|
| $n$ | number of users in the control group |
| $\bar X$ | sample mean — your point estimate of the population mean |
| $\text{median}$ | sample median |
| $s$ | sample standard deviation (use `ddof=1`) |
| $\text{IQR}$ | interquartile range, $Q_3 - Q_1$ (75th percentile $-$ 25th percentile) |
| histogram | with sample **mean** drawn as a solid vertical line and sample **median** as a dashed vertical line |
| 95% CI for $\mu$ | $\bar X \pm 1.96 \cdot s / \sqrt n$, the CLT-based confidence interval for the population mean |

In [ ]:
df = pd.read_csv("ab_data.csv")
control = df["control"].to_numpy()
treatment = df["treatment"].to_numpy()

# Descriptive stats for the CONTROL group
n_ctrl = ...
mean_ctrl = ...
median_ctrl = ...
std_ctrl = ...
iqr_ctrl = ...

# Plot the histogram with mean (solid) and median (dashed) vertical lines.

# 95% CI for the population mean via CLT: x_bar ± 1.96 · s/sqrt(n)
ci_low = ...
ci_high = ...

print(f"point estimate (sample mean) = {mean_ctrl:.4f}")
print(f"95% CI for the mean = [{ci_low:.4f}, {ci_high:.4f}]")

check_answer("2_n", n_ctrl)
check_answer("2_mean", mean_ctrl)
check_answer("2_median", median_ctrl)
check_answer("2_std", std_ctrl)
check_answer("2_iqr", iqr_ctrl)
check_answer("2_ci_low", ci_low)
check_answer("2_ci_high", ci_high)

---

## Part 3 — One-sample hypothesis test

### Primer — the asymptotic Z-test, the $t$-distribution, and SciPy

> In class we built the **asymptotic Z-test**: under $H_0$, the standardized statistic $Z = (\bar X - \mu_0)/(s/\sqrt n)$ is approximately $\mathcal{N}(0, 1)$ by CLT. We do **not** assume the data is Normal — CLT is what justifies the test at large $n$.
>
> SciPy does not ship a Z-test. It ships `ttest_*`. The [$t$-distribution](https://en.wikipedia.org/wiki/Student%27s_t-distribution) $t(\nu)$ is bell-shaped like $\mathcal{N}(0, 1)$ but with slightly heavier tails, parameterized by **degrees of freedom** $\nu$. The t-test technically does assume Normal data; for unknown variance it uses $s$ in place of $\sigma$ and the resulting statistic follows $t(\nu)$ instead of $\mathcal{N}(0, 1)$.
>
> As $\nu \to \infty$ the $t$-distribution converges to $\mathcal{N}(0, 1)$ **very** quickly. At industry $n$ (several dozen and up), the t-test and the asymptotic Z-test give essentially the same number — so we use `scipy.stats.ttest_*` as a numerical stand-in for the asymptotic Z-test we built in class.

### Task 3 — One-sample asymptotic Z-test against a baseline

Last quarter the per-user revenue baseline was $\mu_0 = 10$. Test whether the **control** group from `ab_data.csv` is consistent with that baseline at $\alpha = 0.05$.

$$H_0: \mu = 10, \qquad H_1: \mu \ne 10$$

1. Compute the Z statistic manually: $Z = (\bar X - \mu_0)/(s/\sqrt n)$ with $s$ the sample SD.
2. Compute the two-sided p-value from $\mathcal{N}(0, 1)$: `2 * (1 - st.norm.cdf(|Z|))` or `2 * st.norm.sf(|Z|)`.
3. Repeat with `st.ttest_1samp` — should be almost identical at our $n$.
4. Make the decision.

In [ ]:
mu_0 = 10.0
alpha = 0.05

# Manual Z-test against H_0: μ = μ_0, two-sided
z = ...
p_z = ...

# scipy's t-test for comparison
p_t = ...

# Decision: 1 = reject H_0, 0 = fail to reject
decision = ...

check_answer("3_z_manual", z)
check_answer("3_p_z", p_z)
check_answer("3_p_t", p_t)
check_answer("3_decision", decision)

---

## Part 4 — Two-sample test, two ways

Compare `control` and `treatment`. $H_0: \mu_A = \mu_B$, two-sided, $\alpha = 0.05$.

### Task 4a — Asymptotic Z-test (numerically via `ttest_ind`)

Use `st.ttest_ind(control, treatment, equal_var=False)` — at our $n$ this is numerically equivalent to the asymptotic Z-test from class (see the Part 3 primer). SciPy ships [Welch's variant](https://en.wikipedia.org/wiki/Welch%27s_t-test) which does not assume equal variances; this is what we want for A/B data.

### Task 4b — [Permutation test](https://en.wikipedia.org/wiki/Permutation_test)

A non-parametric alternative. The idea: under $H_0$ the two groups are interchangeable, so we **pool** all observations and re-split them at random many times. For each random split we compute the difference of means. The fraction of random splits where the difference is at least as extreme as what we actually observed is the [permutation p-value](https://en.wikipedia.org/wiki/Permutation_test).

Implement it:

1. Compute `obs_diff = treatment.mean() - control.mean()`.
2. Pool: `pooled = np.concatenate([control, treatment])`.
3. For each of $M = 5000$ iterations:
   - Shuffle the pooled array.
   - Take the first `n_A` as "control*", the rest as "treatment*".
   - Compute `diff_* = treatment*.mean() - control*.mean()`.
4. p-value = fraction of $|\text{diff}_*| \ge |\text{obs\_diff}|$.

Use `np.random.seed(42)` immediately before the loop so the grader can reproduce.

In [ ]:
alpha = 0.05

# 4a — Asymptotic Z-test (computed numerically as Welch t-test via scipy)
z_stat = ...
z_p = ...

# 4b — Permutation test. Implement it following the steps in the markdown above.
# Set np.random.seed(42) immediately before the loop.
M = 5000
perm_p = ...

# Decision: 1 = reject H_0, 0 = fail to reject
decision = ...

check_answer("4_z_stat", z_stat)
check_answer("4_z_p", z_p)
check_answer("4_perm_p", perm_p)
check_answer("4_decision", decision)

---

## Part 5 — Simulation: the p-value distribution

In class we said: under $H_0$, the p-value of a continuous test statistic is $\text{Uniform}[0, 1]$. Let us verify that empirically, then break it.

### Task 5a — Under $H_0$, p-values are uniform; empirical FPR equals $\alpha$

Run $M = 2000$ independent simulated A/B tests where $H_0$ is true by construction. For each:

1. Draw two independent samples of size $n = 200$ from $\mathcal{N}(0, 1)$.
2. Run `st.ttest_ind(..., equal_var=False)`. Store the p-value.

After the loop:

- Plot a histogram of the $M$ p-values (should look flat).
- Compute the empirical false-positive rate at $\alpha = 0.05$: fraction of p-values below $0.05$. Should be close to $0.05$.

In [ ]:
# Use np.random.seed(42) immediately before the simulation loop.
# Run M = 2000 independent A/B tests with H_0 true: both groups from N(0, 1), n = 200 each.
# Use Welch's t-test (scipy.stats.ttest_ind, equal_var=False) for each simulated A/B.
# Plot a histogram of the resulting p-values.

M = 2000
n_sim = 200
alpha = 0.05

# Empirical false-positive rate at α = 0.05
fpr_emp = ...

print(f"empirical FPR = {fpr_emp:.4f}  (target α = {alpha})")

check_answer("5a_fpr", fpr_emp)

### Task 5b — Under $H_1$ true: empirical power (TPR)

Now $H_1$ is true by construction. Same $M = 2000$, $n = 200$, but the treatment is shifted: control $\sim \mathcal{N}(0, 1)$, treatment $\sim \mathcal{N}(0.3, 1)$.

- Plot the p-value histogram (should concentrate near $0$).
- Compute empirical **TPR** (= empirical power) at $\alpha = 0.05$: fraction of p-values below $0.05$.

In [ ]:
# Same setup as 5a, but treatment ~ N(0.3, 1). Use np.random.seed(42).
# Plot the histogram of p-values.

M = 2000
n_sim = 200
alpha = 0.05
true_effect = 0.3

# Empirical TPR at α = 0.05
tpr_emp = ...

print(f"empirical TPR (power) = {tpr_emp:.4f}")

check_answer("5b_tpr", tpr_emp)

### Task 5c — What if the assumptions break?

Repeat Task 5a but draw the data from a heavier-tailed distribution instead of $\mathcal{N}(0, 1)$. Use `np.random.standard_cauchy(n_sim)` for both groups. The Cauchy distribution has no finite variance, so the test's null assumption is violated.

Compute the empirical FPR at $\alpha = 0.05$. Is it close to $0.05$? Why or why not? Write one sentence of explanation in a comment.

In [ ]:
# Same simulation as 5a, but draw the data from `np.random.standard_cauchy(n_sim)` for both groups.
# Use np.random.seed(42). Compute the empirical FPR at α = 0.05.

M = 2000
n_sim = 200
alpha = 0.05

fpr_broken = ...

print(f"empirical FPR with broken assumption = {fpr_broken:.4f}  (nominal α = {alpha})")

# Your one-sentence explanation as a comment:
# ...

check_answer("5c_fpr_broken", fpr_broken)

---

## Part 6 — Bootstrap

### Mini-theory (5 min read)

> **Glivenko–Cantelli.** As $n \to \infty$ the empirical CDF $F_n$ converges uniformly to the true CDF $F$. In words: when we have enough data, the empirical distribution is a faithful representation of the true distribution. We can therefore treat our sample **as if** it were the entire population, and answer "what is the sampling distribution of statistic $T(X)$?" by resampling from it.
>
> **Bootstrap recipe.** Given a sample $X_1, \dots, X_n$:
> 1. Draw a bootstrap resample $X_1^*, \dots, X_n^*$ — same size, **with replacement** from the original sample.
> 2. Compute the statistic of interest $T^* = T(X_1^*, \dots, X_n^*)$ on the resample.
> 3. Repeat $M$ times. The distribution of $T^*$ approximates the sampling distribution of $T$.
> 4. A $(1 - \alpha)$ bootstrap CI is the $\alpha/2$ and $1 - \alpha/2$ percentiles of the resampled $T^*$ values.
>
> The point: no Normal assumption, no closed-form variance, no CLT invocation. The data is the model.

### Task 6a — Bootstrap CI for the mean, vs the CLT CI from Part 2

Build a 95% bootstrap CI for the **mean** of the control group using $M = 5000$ resamples. Print it next to the CLT-based CI you computed in Part 2 — at our sample size the two should be close. Bootstrap is doing the same job without the Normal assumption.

1. `np.random.seed(42)`.
2. For $M$ iterations: sample `n_ctrl` observations from `control` with replacement (`np.random.choice(control, size=n_ctrl, replace=True)`), compute the mean of the resample, store it.
3. Take the $2.5\%$ and $97.5\%$ percentiles of the stored means.

In [ ]:
# Use np.random.seed(42). Draw M = 5000 bootstrap resamples of the control group,
# compute the mean of each, take the 2.5% and 97.5% percentiles as the CI bounds.

M = 5000

boot_ci_low = ...
boot_ci_high = ...

print(f"Bootstrap 95% CI for mean(control) = [{boot_ci_low:.4f}, {boot_ci_high:.4f}]")
print(f"CLT-based CI from Part 2          = [{ci_low:.4f}, {ci_high:.4f}]")

check_answer("6a_ci_low", boot_ci_low)
check_answer("6a_ci_high", boot_ci_high)

### Task 6b — Bonus: bootstrap CI for the median

The mean has a closed-form CLT-based CI (Part 2). The **median** does not — there is no clean formula for the standard error of the sample median without distribution assumptions. Bootstrap handles it the same way:

1. Resample with replacement $M = 5000$ times.
2. Compute the median of each resample.
3. Take the $2.5\%$ and $97.5\%$ percentiles.

This is where bootstrap shines: any statistic, no derivation required, just the data.

In [ ]:
# Use np.random.seed(42). Bootstrap M = 5000 resamples, compute the MEDIAN of each,
# take the 2.5% and 97.5% percentiles.

M = 5000

median_ci_low = ...
median_ci_high = ...

print(f"Bootstrap 95% CI for median(control) = [{median_ci_low:.4f}, {median_ci_high:.4f}]")

check_answer("6c_median_ci_low", median_ci_low)
check_answer("6c_median_ci_high", median_ci_high)

---

## Done

Every `check_answer(...)` above should print **PASS**. Plots should render. Submission: this notebook, with outputs preserved, via Google Classroom.